# Advanced Problems with Solutions: How Python Imports Modules

This notebook contains advanced, runnable problems about Python's import system.

Topics covered:
- runtime imports;
- `sys.path` and module search paths;
- `sys.modules` caching;
- module execution during import;
- custom module loading with `compile` and `exec`;
- safe manipulation of import state;
- simplified custom importers.

Best practices used throughout:
- use temporary directories for import experiments;
- restore `sys.path` after mutation;
- restore `sys.modules` after mutation;
- use assertions for verification;
- avoid polluting the real Python environment.

## Problem 1 — Prove that imports happen at runtime

Create a module file dynamically while the program is already running. Then import it successfully. This proves that Python does not need all modules to exist before the program starts.

In [1]:
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "runtime_created_module"
old_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)

        # The module file does not exist until this line runs.
        (tmp_path / f"{module_name}.py").write_text(
            "VALUE = 42\n"
            "MESSAGE = 'Imported at runtime'\n",
            encoding="utf-8"
        )

        sys.path.insert(0, str(tmp_path))
        importlib.invalidate_caches()

        mod = importlib.import_module(module_name)

        assert mod.VALUE == 42
        assert mod.MESSAGE == "Imported at runtime"
        assert module_name in sys.modules
        assert sys.modules[module_name] is mod
finally:
    sys.path[:] = old_sys_path
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: module was created and imported at runtime.")

Passed: module was created and imported at runtime.


## Problem 2 — Show that `sys.path` controls where Python searches

Create a temporary module in a directory that is not initially importable. Verify that importing fails before adding the directory to `sys.path`, then succeeds afterward.

In [2]:
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "path_search_demo"
old_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    sys.modules.pop(module_name, None)

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        (tmp_path / f"{module_name}.py").write_text("FOUND = True\n", encoding="utf-8")

        assert str(tmp_path) not in sys.path

        try:
            importlib.import_module(module_name)
            raise AssertionError("Import should have failed before adding tmp_path to sys.path")
        except ModuleNotFoundError:
            pass

        sys.path.insert(0, str(tmp_path))
        importlib.invalidate_caches()

        mod = importlib.import_module(module_name)

        assert mod.FOUND is True
        assert Path(mod.__file__).parent == tmp_path
finally:
    sys.path[:] = old_sys_path
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: import search depended on sys.path.")

Passed: import search depended on sys.path.


## Problem 3 — Prove that module code executes during import

Create a module that writes to an external list when imported. Import it and prove that the top-level module code was executed.

In [3]:
import builtins
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "execution_demo"
marker_name = "_execution_demo_events"

old_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules
had_marker = hasattr(builtins, marker_name)
old_marker = getattr(builtins, marker_name, None)

try:
    setattr(builtins, marker_name, [])

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        (tmp_path / f"{module_name}.py").write_text(
            "import builtins\n"
            f"builtins.{marker_name}.append('module code executed')\n"
            "X = 10\n",
            encoding="utf-8"
        )

        sys.path.insert(0, str(tmp_path))
        importlib.invalidate_caches()

        assert getattr(builtins, marker_name) == []

        mod = importlib.import_module(module_name)

        assert mod.X == 10
        assert getattr(builtins, marker_name) == ["module code executed"]
finally:
    sys.path[:] = old_sys_path
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

    if had_marker:
        setattr(builtins, marker_name, old_marker)
    else:
        delattr(builtins, marker_name)

print("Passed: top-level module code executed during import.")

Passed: top-level module code executed during import.


## Problem 4 — Show that repeated imports use `sys.modules`

Create a module whose top-level code increments a counter. Import it twice. Prove that the counter only increments once because the second import uses the cached module object.

In [4]:
import builtins
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "cache_counter_demo"
counter_name = "_cache_counter_demo_count"

old_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules
had_counter = hasattr(builtins, counter_name)
old_counter = getattr(builtins, counter_name, None)

try:
    setattr(builtins, counter_name, 0)

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        (tmp_path / f"{module_name}.py").write_text(
            "import builtins\n"
            f"builtins.{counter_name} += 1\n"
            "VALUE = 'loaded'\n",
            encoding="utf-8"
        )

        sys.path.insert(0, str(tmp_path))
        importlib.invalidate_caches()

        first = importlib.import_module(module_name)
        second = importlib.import_module(module_name)

        assert first is second
        assert getattr(builtins, counter_name) == 1
        assert sys.modules[module_name] is first
finally:
    sys.path[:] = old_sys_path
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

    if had_counter:
        setattr(builtins, counter_name, old_counter)
    else:
        delattr(builtins, counter_name)

print("Passed: repeated import used sys.modules cache.")

Passed: repeated import used sys.modules cache.


## Problem 5 — Demonstrate that `sys.modules` can contain non-module objects

Insert a function into `sys.modules` under a fake module name. Import that name and prove that Python returns the cached object directly.

This is intentionally dangerous and should only be done in isolated experiments.

In [5]:
import importlib
import sys

module_name = "fake_cached_import"
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    fake_object = lambda: "I came from sys.modules"
    sys.modules[module_name] = fake_object

    imported_object = importlib.import_module(module_name)

    assert imported_object is fake_object
    assert imported_object() == "I came from sys.modules"
    assert not hasattr(imported_object, "__dict__") or callable(imported_object)
finally:
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: import returned the object already stored in sys.modules.")

Passed: import returned the object already stored in sys.modules.


## Problem 6 — Build a simplified source-file importer

Implement a function `simple_import_from_path(module_name, file_path)` that:

1. creates a new module object;
2. stores metadata such as `__file__`;
3. reads source code from the file;
4. compiles it;
5. executes it in the module namespace;
6. stores the module in `sys.modules`;
7. returns the module object.

In [6]:
import sys
import tempfile
import types
from pathlib import Path

def simple_import_from_path(module_name, file_path):
    file_path = Path(file_path)

    if module_name in sys.modules:
        return sys.modules[module_name]

    module = types.ModuleType(module_name)
    module.__file__ = str(file_path)

    source = file_path.read_text(encoding="utf-8")
    code = compile(source, str(file_path), "exec")

    sys.modules[module_name] = module
    exec(code, module.__dict__)

    return module

module_name = "simple_loaded_module"
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    sys.modules.pop(module_name, None)

    with tempfile.TemporaryDirectory() as tmp:
        file_path = Path(tmp) / f"{module_name}.py"
        file_path.write_text(
            "A = 10\n"
            "B = 20\n"
            "def add():\n"
            "    return A + B\n",
            encoding="utf-8"
        )

        mod = simple_import_from_path(module_name, file_path)

        assert isinstance(mod, types.ModuleType)
        assert mod.__name__ == module_name
        assert mod.__file__ == str(file_path)
        assert mod.add() == 30
        assert sys.modules[module_name] is mod
finally:
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: custom importer loaded, compiled, executed, cached, and returned a module.")

Passed: custom importer loaded, compiled, executed, cached, and returned a module.


## Problem 7 — Prove that module globals are stored in `module.__dict__`

Use your simplified importer to load a module. Prove that top-level variables and functions become entries in the module namespace dictionary.

In [7]:
import sys
import tempfile
from pathlib import Path

module_name = "namespace_from_exec_demo"
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    sys.modules.pop(module_name, None)

    with tempfile.TemporaryDirectory() as tmp:
        file_path = Path(tmp) / f"{module_name}.py"
        file_path.write_text(
            "PI = 3.14159\n"
            "def area(radius):\n"
            "    return PI * radius ** 2\n",
            encoding="utf-8"
        )

        mod = simple_import_from_path(module_name, file_path)

        assert mod.PI == mod.__dict__["PI"]
        assert mod.area is mod.__dict__["area"]
        assert mod.area(2) == 12.56636
finally:
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: executed module code populated module.__dict__.")

Passed: executed module code populated module.__dict__.


## Problem 8 — Handle failed imports safely

A robust importer should avoid leaving a broken module in `sys.modules` if executing the module code raises an exception.

Write `safer_import_from_path` so that it removes the module from `sys.modules` when execution fails.

In [8]:
import sys
import tempfile
import types
from pathlib import Path

def safer_import_from_path(module_name, file_path):
    file_path = Path(file_path)

    if module_name in sys.modules:
        return sys.modules[module_name]

    module = types.ModuleType(module_name)
    module.__file__ = str(file_path)

    source = file_path.read_text(encoding="utf-8")
    code = compile(source, str(file_path), "exec")

    sys.modules[module_name] = module

    try:
        exec(code, module.__dict__)
    except Exception:
        sys.modules.pop(module_name, None)
        raise

    return module

module_name = "broken_import_demo"
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    sys.modules.pop(module_name, None)

    with tempfile.TemporaryDirectory() as tmp:
        file_path = Path(tmp) / f"{module_name}.py"
        file_path.write_text(
            "STARTED = True\n"
            "raise RuntimeError('boom during import')\n"
            "FINISHED = True\n",
            encoding="utf-8"
        )

        try:
            safer_import_from_path(module_name, file_path)
            raise AssertionError("Import should have failed")
        except RuntimeError as exc:
            assert str(exc) == "boom during import"

        assert module_name not in sys.modules
finally:
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: failed import did not leave a broken module in sys.modules.")

Passed: failed import did not leave a broken module in sys.modules.


## Problem 9 — Compare cached import vs forced reload by deletion

Create a module file with one value, import it, change the file, import again, and prove that the cached object is returned. Then delete the entry from `sys.modules`, import again, and prove that the new file contents are used.

In [9]:
import importlib
import sys
import tempfile
import time
from pathlib import Path

module_name = "cache_vs_file_demo"
old_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    sys.modules.pop(module_name, None)

    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        file_path = tmp_path / f"{module_name}.py"

        file_path.write_text("VERSION = 1\n", encoding="utf-8")
        sys.path.insert(0, str(tmp_path))
        importlib.invalidate_caches()

        first = importlib.import_module(module_name)
        assert first.VERSION == 1

        # Ensure timestamp changes on file systems with coarse timestamp resolution.
        time.sleep(1.1)
        file_path.write_text("VERSION = 2\n", encoding="utf-8")
        importlib.invalidate_caches()

        second = importlib.import_module(module_name)
        assert second is first
        assert second.VERSION == 1

        sys.modules.pop(module_name)
        third = importlib.import_module(module_name)

        assert third is not first
        assert third.VERSION == 2
finally:
    sys.path[:] = old_sys_path
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: normal import used cache; deleting sys.modules forced a fresh import.")

Passed: normal import used cache; deleting sys.modules forced a fresh import.


## Problem 10 — Implement an import-state context manager

Create a context manager that temporarily adds a directory to `sys.path` and restores selected `sys.modules` entries afterward. Use it to safely import a temporary module.

In [10]:
from contextlib import contextmanager
import importlib
import sys
import tempfile
from pathlib import Path

@contextmanager
def isolated_import_environment(path, *module_names):
    old_sys_path = sys.path.copy()
    old_modules = {name: sys.modules.get(name) for name in module_names}
    had_modules = {name: name in sys.modules for name in module_names}

    try:
        sys.path.insert(0, str(path))
        importlib.invalidate_caches()
        yield
    finally:
        sys.path[:] = old_sys_path

        for name in module_names:
            if had_modules[name]:
                sys.modules[name] = old_modules[name]
            else:
                sys.modules.pop(name, None)

        importlib.invalidate_caches()

module_name = "isolated_module"

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    (tmp_path / f"{module_name}.py").write_text("STATUS = 'safe import'\n", encoding="utf-8")

    with isolated_import_environment(tmp_path, module_name):
        mod = importlib.import_module(module_name)
        assert mod.STATUS == "safe import"
        assert module_name in sys.modules

    assert module_name not in sys.modules

print("Passed: isolated import environment restored sys.path and sys.modules.")

Passed: isolated import environment restored sys.path and sys.modules.


## Problem 11 — Detect module shadowing caused by `sys.path` order

Create two different directories containing modules with the same name. Put both directories on `sys.path` in different orders and prove that Python imports the first matching module it finds.

In [11]:
import importlib
import sys
import tempfile
from pathlib import Path

module_name = "duplicate_name_demo"
old_sys_path = sys.path.copy()
old_module = sys.modules.get(module_name)
had_module = module_name in sys.modules

try:
    sys.modules.pop(module_name, None)

    with tempfile.TemporaryDirectory() as tmp1, tempfile.TemporaryDirectory() as tmp2:
        path1 = Path(tmp1)
        path2 = Path(tmp2)

        (path1 / f"{module_name}.py").write_text("ORIGIN = 'first directory'\n", encoding="utf-8")
        (path2 / f"{module_name}.py").write_text("ORIGIN = 'second directory'\n", encoding="utf-8")

        sys.path.insert(0, str(path2))
        sys.path.insert(0, str(path1))
        importlib.invalidate_caches()

        mod1 = importlib.import_module(module_name)
        assert mod1.ORIGIN == "first directory"

        sys.modules.pop(module_name)
        sys.path[:] = old_sys_path.copy()
        sys.path.insert(0, str(path1))
        sys.path.insert(0, str(path2))
        importlib.invalidate_caches()

        mod2 = importlib.import_module(module_name)
        assert mod2.ORIGIN == "second directory"
finally:
    sys.path[:] = old_sys_path
    if had_module:
        sys.modules[module_name] = old_module
    else:
        sys.modules.pop(module_name, None)

print("Passed: sys.path order determined which same-named module was imported.")

Passed: sys.path order determined which same-named module was imported.


## Problem 12 — Build a tiny import diagnostic tool

Write a function `diagnose_import(name)` that reports whether a module is already cached, whether it can be found by Python's import machinery, and where it would likely be loaded from.

In [12]:
import importlib.util
import sys

def diagnose_import(name):
    spec = importlib.util.find_spec(name)

    return {
        "name": name,
        "cached_in_sys_modules": name in sys.modules,
        "found_by_import_system": spec is not None,
        "origin": None if spec is None else spec.origin,
        "loader_type": None if spec is None or spec.loader is None else type(spec.loader).__name__
    }

math_report = diagnose_import("math")
missing_report = diagnose_import("definitely_not_a_real_module_12345")

assert math_report["found_by_import_system"] is True
assert math_report["origin"] is not None
assert missing_report["found_by_import_system"] is False
assert missing_report["origin"] is None

print("math report:", math_report)
print("missing report:", missing_report)
print("Passed: diagnostic tool inspected cache and import discovery.")

math report: {'name': 'math', 'cached_in_sys_modules': True, 'found_by_import_system': True, 'origin': 'built-in', 'loader_type': 'type'}
missing report: {'name': 'definitely_not_a_real_module_12345', 'cached_in_sys_modules': False, 'found_by_import_system': False, 'origin': None, 'loader_type': None}
Passed: diagnostic tool inspected cache and import discovery.


# Final Summary

Important conclusions:

1. Python imports modules at runtime.
2. `sys.path` controls where Python searches for importable modules.
3. `sys.modules` is the central import cache.
4. If a name already exists in `sys.modules`, import usually returns that object immediately.
5. Top-level module code executes during import.
6. Module execution populates the module object's `__dict__`.
7. A simplified importer can be built with `types.ModuleType`, `compile`, and `exec`.
8. Import experiments should restore global state such as `sys.path` and `sys.modules`.
9. `sys.path` order can cause module shadowing.
10. Failed imports should not leave broken modules cached.